# GAMMA `doppler_2d_SLC`, mosaic last — geometry, corrections & debias

Doppler estimator fixed: GAMMA `SLC_deramp_ScanSAR` → `doppler_2d_SLC` per burst
→ stitch last. `BLSZ = 256`, `GEOM_SOURCE = 'gamma'` (the best-looking choice).

**Re-runnable:** `autoreload` is on, so edited `.py` files are picked up without a
kernel restart. Every figure is shown inline **and** saved to
`plots/gamma_corr_experiment/`, so the images are always there to open.

Steps: (1) geometry-source comparison, (2) component breakdown, (3) correction
sweep, (4) debias the residual offset, (5) sign check.

Heavy results cached under `data/sentinel-1/scene1/gamma_corr_cache/` — delete
that folder to force a recompute after a pipeline-logic change.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys

_root = os.getcwd()
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'scripts')):
    _root = os.path.dirname(_root)
if not os.path.isdir(os.path.join(_root, 'scripts')):
    raise RuntimeError('Could not locate repo root (no scripts/ directory found)')
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print('repo root:', _root)

import pickle
import itertools
import numpy as np
import matplotlib.pyplot as plt

from scripts.sentinel_1.pipeline import run_all_bursts, run_gamma_dop2d_pipeline
from scripts.sentinel_1.gamma_variants import gamma_doppler_mosaic_last
from scripts.sentinel_1.grid_merge import merge_burst_grids, merge_model_grid, compute_stats, smooth_block_grid

## Data paths

In [ ]:
import glob

DATA  = 'data'
SCENE = 'scene1'
S1_DIR = f'{DATA}/sentinel-1/{SCENE}'

SLC_SAFE  = f'{S1_DIR}/S1A_IW_SLC.SAFE'
OCN_SAFE  = f'{S1_DIR}/S1A_IW_OCN.SAFE'
POEORB    = glob.glob(f'{S1_DIR}/S1A_OPER_AUX_POEORB_*.EOF')[0]
AUX_CAL   = f'{DATA}/sentinel-1/S1A_AUX_CAL_V20190228T092500_G20240327T102320.SAFE'
ANNOT_XML = glob.glob(f'{SLC_SAFE}/annotation/s1a-iw1-slc-vv-*.xml')[0]

ERA5_WIND = f'{DATA}/era5_data/{SCENE}/era5_wind.nc'
ERA5_WAVE = f'{DATA}/era5_data/{SCENE}/era5_wave.nc'
GLO12     = f'{DATA}/era5_data/{SCENE}/glo12.nc'

SUBSWATH, POL = 'iw1', 'vv'

_paths = [SLC_SAFE, OCN_SAFE, POEORB, AUX_CAL, ANNOT_XML, ERA5_WIND, ERA5_WAVE, GLO12]
print('all paths exist:', all(os.path.exists(p) for p in _paths))
for p in _paths:
    if not os.path.exists(p):
        print('  MISSING:', p)

## Configuration

In [ ]:
BLSZ        = 256          # GAMMA doppler_2d_SLC azimuth block size
GEOM_SOURCE = 'gamma'     # 'gamma' | 'annotation' | 'poeorb'
GEOM_SOURCES = ['gamma', 'annotation', 'poeorb']
# Block-grid smoothing applied to the chosen-geometry result before
# component breakdown / correction sweep / debias.  Pair with
# GAMMA demod_back step/hanning to suppress burst-boundary stepping
# without per-burst amplitude attenuation.  Set both to 1 to disable.
SMOOTH_AZ = 3
SMOOTH_RG = 1


# Figures are written here every run.
FIGDIR = 'plots/gamma_corr_experiment'
os.makedirs(FIGDIR, exist_ok=True)

def save(fig, name):
    """Save a figure into FIGDIR and report the path."""
    path = f'{FIGDIR}/{name}.png'
    fig.savefig(path, dpi=130, bbox_inches='tight')
    print(f'  saved {path}')

CACHE = f'{S1_DIR}/gamma_corr_cache'
os.makedirs(CACHE, exist_ok=True)

def cached(key, fn):
    """Pickle-cache the return value of fn() under CACHE/key.pkl."""
    path = f'{CACHE}/{key}.pkl'
    if os.path.exists(path):
        print(f'[cache]   {key}')
        with open(path, 'rb') as f:
            return pickle.load(f)
    print(f'[compute] {key} ...')
    obj = fn()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj

## Step 1 — geometry-source comparison

`doppler_2d_SLC` is run once per deramped burst (cached `.npz`); the corrections
pipeline is run once per `geom_source`. Maps show `v_current` (mispointing off)
for each geometry choice against GLO12.

In [ ]:
def _run(geom):
    npz = gamma_doppler_mosaic_last(BLSZ)
    return run_gamma_dop2d_pipeline(
        dop2d_npz=npz, annotation_xml=ANNOT_XML, subswath=SUBSWATH,
        poeorb_path=POEORB, aux_cal_path=AUX_CAL, ocn_safe=OCN_SAFE,
        era5_wind=ERA5_WIND, era5_wave=ERA5_WAVE, glo12=GLO12, polarisation=POL,
        geom_source=geom,
    )

results = {
    g: cached(f'gamma_dop2d_{g}_blsz{BLSZ}', lambda g=g: _run(g))
    for g in GEOM_SOURCES
}
ocn = cached('ocn_product', lambda: run_all_bursts(
    SLC_SAFE, SUBSWATH, POEORB, AUX_CAL, OCN_SAFE,
    ERA5_WIND, ERA5_WAVE, GLO12, POL, use_ocn_dc=True)[0])
print('geometry sources computed:', list(results))

In [ ]:
def grid_var(res, values):
    """Interpolate a block-grid field onto a regular lat/lon grid."""
    tagged = dict(res); tagged['_f'] = np.asarray(values, dtype=np.float32)
    glat, glon, g = merge_burst_grids([tagged], variable='_f', overlap='average')
    return glat, glon, g

def score_field(res, values):
    """Regrid a field + the model, return glat/glon/sar/model/bias/rmse/r."""
    glat, glon, sar = grid_var(res, values)
    _, _, model = merge_model_grid([res])
    bias, rmse, r = compute_stats(sar, model)
    return dict(glat=glat, glon=glon, sar=sar, model=model, bias=bias, rmse=rmse, r=r)

geom_grids = {g: score_field(results[g], results[g]['v_current']) for g in GEOM_SOURCES}

ref = geom_grids[GEOM_SOURCES[0]]
_a  = [gg['sar'][np.isfinite(gg['sar'])] for gg in geom_grids.values()]
_a += [ref['model'][np.isfinite(ref['model'])]]
VG  = float(np.nanpercentile(np.abs(np.concatenate(_a)), 98))
e   = [ref['glon'].min(), ref['glon'].max(), ref['glat'].min(), ref['glat'].max()]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
im = axes[0].imshow(ref['model'], extent=e, origin='lower', cmap='RdBu_r',
                    vmin=-VG, vmax=VG, aspect='auto')
axes[0].set_title('GLO12 model', fontweight='bold')
for ax, g in zip(axes[1:], GEOM_SOURCES):
    gg = geom_grids[g]
    ax.imshow(gg['sar'], extent=e, origin='lower', cmap='RdBu_r',
              vmin=-VG, vmax=VG, aspect='auto')
    ax.set_title(f"geom='{g}'  (v_current)\n"
                 f"bias={gg['bias']:+.3f}  RMSE={gg['rmse']:.3f}  r={gg['r']:+.3f}", fontsize=9)
for ax in axes:
    ax.set_xlabel('Longitude [deg]'); ax.set_ylabel('Latitude [deg]')
fig.colorbar(im, ax=axes.tolist(), label='radial current [m/s]', fraction=0.012, pad=0.02)
fig.suptitle(f'Geometry-source comparison (blsz={BLSZ})', fontsize=12, y=1.04)
save(fig, 'geom_comparison'); plt.show()

print(f"\n{'geom_source':<13}{'v_r mean':>11}{'bias':>9}{'RMSE':>9}{'r':>8}")
for g in GEOM_SOURCES:
    gg = geom_grids[g]
    print(f"{g:<13}{np.nanmean(results[g]['v_r']):>+11.3f}"
          f"{gg['bias']:>+9.3f}{gg['rmse']:>9.3f}{gg['r']:>+8.3f}")

## Step 2 — component breakdown (`GEOM_SOURCE`)

Each current term mapped on its own colour scale, plus the scene-mean of every
component.

In [ ]:
result = results[GEOM_SOURCE]
if SMOOTH_AZ > 1 or SMOOTH_RG > 1:
    result = smooth_block_grid(result, smooth_az=SMOOTH_AZ, smooth_rg=SMOOTH_RG)
    print(f"applied block-grid smoothing az={SMOOTH_AZ} rg={SMOOTH_RG}")
print(f"using geom_source='{GEOM_SOURCE}'")

panels = [
    ('-v_r  (SAR signal)',  -result['v_r']),
    ('+v_miss_ocn',          result['v_miss_ocn']),
    ('-v_stokes',           -result['v_stokes']),
    ('-v_wave',             -result['v_wave']),
    ('GLO12 v_model',        result['v_model']),
    ('v_current (no miss)',  result['v_current']),
]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (name, field) in zip(axes.ravel(), panels):
    glat, glon, g = grid_var(result, field)
    v = float(np.nanpercentile(np.abs(g[np.isfinite(g)]), 98)) if np.isfinite(g).any() else 1.0
    im = ax.imshow(g, extent=[glon.min(), glon.max(), glat.min(), glat.max()],
                   origin='lower', cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
    ax.set_title(f"{name}\nmean={np.nanmean(field):+.3f}  (scale +/-{v:.2f}) m/s", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"Component breakdown — geom_source='{GEOM_SOURCE}', blsz={BLSZ}", fontsize=13, y=1.01)
plt.tight_layout(); save(fig, 'component_breakdown'); plt.show()

terms = {'-v_r': -result['v_r'], '+v_miss_ocn': result['v_miss_ocn'],
         '-v_stokes': -result['v_stokes'], '-v_wave': -result['v_wave'],
         'v_current': result['v_current'], 'GLO12': result['v_model']}
means = {k: float(np.nanmean(v)) for k, v in terms.items()}
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(list(means), list(means.values()),
       color=['tab:blue', 'tab:red', 'tab:green', 'tab:orange', 'k', 'tab:purple'], alpha=0.85)
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel('scene-mean [m/s]')
ax.set_title(f"Component scene-means  (geom_source='{GEOM_SOURCE}')")
for i, k in enumerate(means):
    ax.annotate(f'{means[k]:+.3f}', (i, means[k]),
                ha='center', va='bottom' if means[k] >= 0 else 'top', fontsize=9)
plt.xticks(rotation=15); plt.tight_layout(); save(fig, 'component_means'); plt.show()

## Step 3 — correction sweep (`GEOM_SOURCE`)

Every on/off combination of {mispointing, Stokes, wave}, recombined from stored
components and scored vs GLO12, sorted by RMSE.

In [ ]:
def recombine(res, mispointing, stokes, wave, vr_coeff=-1.0):
    v = vr_coeff * np.asarray(res['v_r'], dtype=np.float64)
    if mispointing: v = v + res['v_miss_ocn']
    if stokes:      v = v - res['v_stokes']
    if wave:        v = v - res['v_wave']
    return v.astype(np.float32)

def score(res, **flags):
    return score_field(res, recombine(res, **flags))

def combo_label(m, s, w):
    return ' '.join(['-v_r'] + (['+miss'] if m else [])
                    + (['-stokes'] if s else []) + (['-wave'] if w else []))

SWEEP = []
for m, s, w in itertools.product([True, False], repeat=3):
    g = score(result, mispointing=m, stokes=s, wave=w)
    g['label'] = combo_label(m, s, w)
    SWEEP.append(g)
SWEEP.sort(key=lambda g: g['rmse'])

ocn_g = score_field(ocn, ocn['v_current_ocn'])
_all = [g['sar'][np.isfinite(g['sar'])] for g in SWEEP]
_all += [SWEEP[0]['model'][np.isfinite(SWEEP[0]['model'])]]
VMAX = float(np.nanpercentile(np.abs(np.concatenate(_all)), 98))

tiles = [('GLO12 model', SWEEP[0]['model'], None), ('OCN product', ocn_g['sar'], None)]
tiles += [(g['label'], g['sar'], (g['bias'], g['rmse'], g['r'])) for g in SWEEP]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for ax, (name, g, st) in zip(axes, tiles):
    im = ax.imshow(g, extent=e, origin='lower', cmap='RdBu_r',
                   vmin=-VMAX, vmax=VMAX, aspect='auto')
    ax.set_title(name if st is None else
                 f'{name}\nbias={st[0]:+.3f} RMSE={st[1]:.3f} r={st[2]:+.3f}', fontsize=9)
for ax in axes[len(tiles):]:
    ax.axis('off')
fig.colorbar(im, ax=axes.tolist(), label='radial current [m/s]', fraction=0.015, pad=0.02)
fig.suptitle(f"Correction sweep — geom_source='{GEOM_SOURCE}', blsz={BLSZ} (sorted by RMSE)",
             fontsize=13, y=1.005)
save(fig, 'correction_sweep'); plt.show()

print(f"{'correction set':<26}{'bias':>9}{'RMSE':>9}{'r':>8}")
for g in SWEEP:
    print(f"{g['label']:<26}{g['bias']:>+9.3f}{g['rmse']:>9.3f}{g['r']:>+8.3f}")

## Step 4 — debias the residual offset

With `geom='gamma'` the SAR carries the current *pattern* but not its absolute
level. Remove the constant offset of the best correction set so RMSE/r reflect
pattern agreement, and report the offset on its own.

In [ ]:
best = SWEEP[0]                                 # lowest-RMSE correction set
offset = float(np.nanmean(best['sar']) - np.nanmean(best['model']))
deb = best['sar'] - offset
bias_d, rmse_d, r_d = compute_stats(deb, best['model'])
lam_half = results[GEOM_SOURCE]['wavelength_m'] / 2.0

print(f"variant         : {best['label']}  (geom_source='{GEOM_SOURCE}')")
print(f"residual offset : {offset:+.3f} m/s  (~{offset / lam_half:+.1f} Hz)")
print(f"before debias   : bias={best['bias']:+.3f}  RMSE={best['rmse']:.3f}  r={best['r']:+.3f}")
print(f"after  debias   : bias={bias_d:+.3f}  RMSE={rmse_d:.3f}  r={r_d:+.3f}")

vmx = float(np.nanpercentile(np.abs(best['model'][np.isfinite(best['model'])]), 98))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, (name, data) in zip(ax, [('GLO12 model', best['model']),
                                (f"SAR {best['label']} (raw)", best['sar']),
                                (f"SAR {best['label']} (debiased {offset:+.2f})", deb)]):
    im = a.imshow(data, extent=e, origin='lower', cmap='RdBu_r',
                  vmin=-vmx, vmax=vmx, aspect='auto')
    a.set_title(name, fontsize=10)
    a.set_xlabel('Longitude [deg]'); a.set_ylabel('Latitude [deg]')
fig.colorbar(im, ax=ax.tolist(), label='radial current [m/s]', fraction=0.015, pad=0.02)
fig.suptitle(f"Debias — geom='{GEOM_SOURCE}'  offset={offset:+.3f} m/s", fontsize=13, y=1.03)
save(fig, 'debias'); plt.show()

## Step 5 — sign / convention check

`-v_r` vs `+v_r` (no metocean corrections). If `+v_r` correlates clearly
better, a radial-velocity or look-direction sign is flipped.

In [ ]:
checks = [
    ('-v_r (standard)', score(result, mispointing=False, stokes=False, wave=False, vr_coeff=-1.0)),
    ('+v_r (flipped)',  score(result, mispointing=False, stokes=False, wave=False, vr_coeff=+1.0)),
]
vmx = max(float(np.nanpercentile(np.abs(g['sar'][np.isfinite(g['sar'])]), 98)) for _, g in checks)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
g0 = checks[0][1]
ax[0].imshow(g0['model'], extent=e, origin='lower', cmap='RdBu_r',
             vmin=-vmx, vmax=vmx, aspect='auto')
ax[0].set_title('GLO12 model', fontweight='bold')
for a, (name, g) in zip(ax[1:], checks):
    im = a.imshow(g['sar'], extent=e, origin='lower', cmap='RdBu_r',
                  vmin=-vmx, vmax=vmx, aspect='auto')
    a.set_title(f"{name}\nbias={g['bias']:+.3f} RMSE={g['rmse']:.3f} r={g['r']:+.3f}", fontsize=10)
fig.colorbar(im, ax=ax.tolist(), label='radial current [m/s]', fraction=0.015, pad=0.02)
fig.suptitle(f"Sign / convention check (geom='{GEOM_SOURCE}')", fontsize=13, y=1.03)
save(fig, 'sign_check'); plt.show()
for name, g in checks:
    print(f"  {name:18s} bias={g['bias']:+.3f}  RMSE={g['rmse']:.3f}  r={g['r']:+.3f}")

## Notes

All figures from this run are in `plots/gamma_corr_experiment/`:
`geom_comparison`, `component_breakdown`, `component_means`, `correction_sweep`,
`debias`, `sign_check`.

`autoreload` is active, so after a `.py` edit just re-run the cells — no kernel
restart. If a *pipeline-logic* change should change the numbers, also delete
`data/sentinel-1/scene1/gamma_corr_cache/` so the results recompute.